In [1]:
import pandas as pd
import sqlite3
import os 

In [2]:
raw_data_path= "C:/Users/rs38129/Desktop/BI/retail-operations-analytics/Data/Raw"
db_path="C:/Users/rs38129/Desktop/BI/retail-operations-analytics/Data/retail_olist.db"

In [3]:
customers=pd.read_csv("C:/Users/rs38129/Desktop/BI/retail-operations-analytics/Data/Raw/olist_customers_dataset.csv")
geolocation=pd.read_csv("C:/Users/rs38129/Desktop/BI/retail-operations-analytics/Data/Raw/olist_geolocation_dataset.csv")
order_items=pd.read_csv("C:/Users/rs38129/Desktop/BI/retail-operations-analytics/Data/Raw/olist_order_items_dataset.csv")
order_payments=pd.read_csv("C:/Users/rs38129/Desktop/BI/retail-operations-analytics/Data/Raw/olist_order_payments_dataset.csv")
order_reviews=pd.read_csv("C:/Users/rs38129/Desktop/BI/retail-operations-analytics/Data/Raw/olist_order_reviews_dataset.csv")
orders=pd.read_csv("C:/Users/rs38129/Desktop/BI/retail-operations-analytics/Data/Raw/olist_orders_dataset.csv")
products=pd.read_csv("C:/Users/rs38129/Desktop/BI/retail-operations-analytics/Data/Raw/olist_products_dataset.csv")
sellers=pd.read_csv("C:/Users/rs38129/Desktop/BI/retail-operations-analytics/Data/Raw/olist_sellers_dataset.csv")
category_translation=pd.read_csv("C:/Users/rs38129/Desktop/BI/retail-operations-analytics/Data/Raw/product_category_name_translation.csv")

In [4]:
tables = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}
for name, df in tables.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

customers: 99441 rows, 5 columns
geolocation: 1000163 rows, 5 columns
order_items: 112650 rows, 7 columns
order_payments: 103886 rows, 5 columns
order_reviews: 99224 rows, 7 columns
orders: 99441 rows, 8 columns
products: 32951 rows, 9 columns
sellers: 3095 rows, 4 columns
category_translation: 71 rows, 2 columns


In [5]:
connection=sqlite3.connect(db_path)
for name, df in tables.items():
    df.to_sql(name, connection, if_exists="replace", index=False)
    print(f"Loaded: {name}")


Loaded: customers
Loaded: geolocation
Loaded: order_items
Loaded: order_payments
Loaded: order_reviews
Loaded: orders
Loaded: products
Loaded: sellers
Loaded: category_translation


In [6]:
cursor=connection.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print("Tables in database:")
for table in cursor.fetchall():
    print("-", table[0])

Tables in database:
- customers
- geolocation
- order_items
- order_payments
- order_reviews
- orders
- products
- sellers
- category_translation


In [7]:
connection.close()

In [8]:
connection=sqlite3.connect(db_path)
df_orders=pd.read_sql("SELECT * FROM orders LIMIT 5", connection)
print(df_orders.columns.tolist())
df_orders.head()

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [9]:
df_dates = pd.read_sql("""
    SELECT 
        MIN(order_purchase_timestamp) AS earliest_order,
        MAX(order_purchase_timestamp) AS latest_order,
        COUNT(*) AS total_orders
    FROM orders
""", connection)
df_dates

,earliest_order,latest_order,total_orders
0,2016-09-04 21:15:19,2018-10-17 17:30:18,99441


In [10]:
df_status=pd.read_sql("""
    SELECT
        order_status, COUNT(*) AS count FROM orders
        GROUP BY order_status
        ORDER BY count DESC
""", connection)
df_status

,order_status,count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2
